In [14]:
# =========================
# 1. Import Libraries
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

import joblib

# =========================
# 2. Load Dataset
# =========================
df = pd.read_csv("kidney_disease_dataset.csv")

df.replace('?', np.nan, inplace=True)
df.columns = df.columns.str.strip()

# =========================
# 3. Fix Target (Binary)
# =========================
df['Target'] = df['Target'].astype(str).str.strip().str.lower()

df['Target'] = df['Target'].apply(
    lambda x: 0 if x == 'no_disease' else 1
)

# =========================
# 4. Select BEST 10 Features
# =========================
features = [
    'Serum creatinine (mg/dl)',
    'Blood urea (mg/dl)',
    'Estimated Glomerular Filtration Rate (eGFR)',
    'Hemoglobin level (gms)',
    'Albumin in urine',
    'Specific gravity of urine',
    'Blood pressure (mm/Hg)',
    'Age of the patient',
    'Diabetes mellitus (yes/no)',
    'Hypertension (yes/no)'
]

X = df[features]
y = df['Target']

# =========================
# 5. Separate Types
# =========================
numeric_features = [
    'Serum creatinine (mg/dl)',
    'Blood urea (mg/dl)',
    'Estimated Glomerular Filtration Rate (eGFR)',
    'Hemoglobin level (gms)',
    'Albumin in urine',
    'Specific gravity of urine',
    'Blood pressure (mm/Hg)',
    'Age of the patient'
]

categorical_features = [
    'Diabetes mellitus (yes/no)',
    'Hypertension (yes/no)'
]

# =========================
# 6. Preprocessing
# =========================
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

# =========================
# 7. Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 8. Model (High Accuracy)
# =========================
model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=2,
        random_state=42,
        eval_metric='logloss'
    ))
])

# =========================
# 9. Train
# =========================
model.fit(X_train, y_train)

# =========================
# 10. Evaluate
# =========================
y_pred = model.predict(X_test)

print("\n🔥 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Report:\n", classification_report(y_test, y_pred))

# =========================
# 11. Save Model
# =========================
joblib.dump(model, "kidney_disease_pipeline.pkl")

print("\n✅ Model saved as ckd_top10_model.pkl")


🔥 Accuracy: 0.7908958130477117

📊 Report:
               precision    recall  f1-score   support

           0       0.80      0.98      0.88      3287
           1       0.23      0.02      0.04       821

    accuracy                           0.79      4108
   macro avg       0.51      0.50      0.46      4108
weighted avg       0.69      0.79      0.71      4108


✅ Model saved as ckd_top10_model.pkl
